# שבוע 11: סטטיסטיקה מתקדמת למורפומטריקה

שיעור זה מעמיק את הכלים הסטטיסטיים:
- ויזואליזציה של התפלגות ה-MANOVA
- אלומטריה עם רגרסיה לינארית
- השוואות מרובות ותיקון בונפרוני
- מפת חום של ערכי p

In [ ]:
!pip install morphops python-bidi scipy -q

import numpy as np
import matplotlib
matplotlib.rcParams['font.family'] = 'DejaVu Sans'
matplotlib.rcParams['axes.unicode_minus'] = False
import matplotlib.pyplot as plt
from bidi.algorithm import get_display
import morphops as mops
from scipy import stats
rtl = get_display
print('הכל מוכן!')

In [ ]:
def parse_efa_dat(text_or_path):
    if '\n' in text_or_path:
        lines = text_or_path.strip().split('\n')
    else:
        with open(text_or_path, encoding='utf-8', errors='replace') as f:
            lines = f.readlines()
    groups, data, sizes = [], [], []
    for line in lines[2:]:
        line = line.strip() if hasattr(line, 'strip') else line
        if not line:
            continue
        cols = line.split('\t')
        groups.append(cols[3].strip())
        sizes.append(float(cols[4].replace(',', '.').replace('E', 'e')))
        coeffs = []
        for c in cols[5:]:
            c = c.strip()
            if c and c != '-':
                try:
                    coeffs.append(float(c.replace(',', '.').replace('E', 'e')))
                except:
                    pass
        data.append(coeffs)
    min_len = min(len(d) for d in data)
    return np.array([d[:min_len] for d in data]), np.array(groups), np.array(sizes)

print('parse_efa_dat מוכן')

In [ ]:
import urllib.request

AXES_URL = 'https://raw.githubusercontent.com/shaigordin/comparch/2026/morphometrics/data/axes/axes_efa.dat'

try:
    with urllib.request.urlopen(AXES_URL, timeout=15) as r:
        text = r.read().decode('utf-8', errors='replace')
    efa_data, groups, sizes = parse_efa_dat(text)
    print(f'נטענו {len(efa_data)} קרדומות, {efa_data.shape[1]} מקדמים')
except Exception as e:
    print(f'שגיאה: {e} — משתמשים בנתונים סינתטיים')
    np.random.seed(42)
    base = np.random.randn(60, 120) * 0.1
    base[:30, 0] += 0.5
    base[30:, 2] += 0.4
    efa_data = base
    groups = np.array(['G3']*30 + ['G4']*30)
    sizes = np.random.uniform(50, 150, 60)
    sizes[:30] += 20

from sklearn.decomposition import PCA
pca_ax = PCA()
scores_ax = pca_ax.fit_transform(efa_data)
ve_ax = pca_ax.explained_variance_ratio_ * 100
print(f'PCA: PC1={ve_ax[0]:.1f}%, PC2={ve_ax[1]:.1f}%')

## MANOVA הסתברותי עם ויזואליזציה של התפלגות ה-F

נציג את התפלגות ה-F תחת H₀ (אקראיות) ואת הערך הנצפה.

In [ ]:
def permutation_manova(X, groups, n_perm=999, seed=42):
    np.random.seed(seed)
    def f_stat(X, g):
        ug = np.unique(g)
        gm = X.mean(axis=0)
        between = sum(np.sum(g==u) * np.sum((X[g==u].mean(0) - gm)**2) for u in ug)
        within  = sum(np.sum((X[g==u] - X[g==u].mean(0))**2) for u in ug)
        return between / within if within > 0 else 0
    obs = f_stat(X, groups)
    perm = [f_stat(X, np.random.permutation(groups)) for _ in range(n_perm)]
    p = (np.sum(np.array(perm) >= obs) + 1) / (n_perm + 1)
    return obs, p, obs / (obs + 1), np.array(perm)

F_obs, p_val, r_sq, perm_dist = permutation_manova(scores_ax[:, :4], groups)

fig, ax = plt.subplots(figsize=(9, 5))
ax.hist(perm_dist, bins=40, color='lightsteelblue', edgecolor='white',
        label=rtl('התפלגות F תחת H₀ (999 תמורות)'))
ax.axvline(F_obs, color='red', lw=2.5, linestyle='--',
           label=f'F נצפה = {F_obs:.3f}  (p = {p_val:.4f})')
ax.fill_betweenx([0, ax.get_ylim()[1] if ax.get_ylim()[1] > 0 else 10],
                  F_obs, perm_dist.max(), alpha=0.2, color='red',
                  label=f'p = {p_val:.4f}')
ax.set_xlabel('F-statistic', fontsize=12)
ax.set_ylabel(rtl('תדירות'), fontsize=12)
ax.set_title(rtl('התפלגות MANOVA הסתברותי — קרדומות G3/G4'), fontsize=13)
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

print(f'F = {F_obs:.4f}, p = {p_val:.4f}, R² = {r_sq:.4f}')

## אלומטריה: גודל vs. PC1

In [ ]:
log_sizes = np.log(sizes)
slope, intercept, r_value, p_allo, se = stats.linregress(log_sizes, scores_ax[:, 0])

fig, ax = plt.subplots(figsize=(8, 6))
for grp, color in [('G3', 'steelblue'), ('G4', 'darkorange')]:
    mask = groups == grp
    ax.scatter(log_sizes[mask], scores_ax[mask, 0],
               color=color, s=70, alpha=0.8, label=grp, zorder=3)

x_line = np.linspace(log_sizes.min(), log_sizes.max(), 100)
ax.plot(x_line, slope * x_line + intercept, 'k--', lw=2,
        label=f'רגרסיה: R²={r_value**2:.3f}, p={p_allo:.4f}')

# רווח בטחון 95%
n = len(log_sizes)
se_line = se * np.sqrt(1/n + (x_line - log_sizes.mean())**2 / np.sum((log_sizes - log_sizes.mean())**2))
t_crit = stats.t.ppf(0.975, df=n-2)
ax.fill_between(x_line,
                slope*x_line + intercept - t_crit*se_line,
                slope*x_line + intercept + t_crit*se_line,
                alpha=0.15, color='gray', label='רווח בטחון 95%')

ax.set_xlabel(rtl('log(גודל קרדום)'), fontsize=11)
ax.set_ylabel(f'PC1 ({ve_ax[0]:.1f}%)', fontsize=11)
ax.set_title(rtl('אלומטריה: קשר בין גודל לצורה'), fontsize=13)
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

print(f'שיפוע: {slope:.4f}')
print(f'R²: {r_value**2:.4f}')
print(f'p: {p_allo:.4f}')
if p_allo < 0.05:
    print('** יש קשר מובהק בין גודל לצורה **')
else:
    print('אין קשר מובהק בין גודל לצורה')

## השוואות מרובות ותיקון בונפרוני

כאשר בודקים מספר השוואות בו-זמנית, עולה הסיכוי לשגיאת סוג I.  
**תיקון בונפרוני**: מחלקים את סף המובהקות (α=0.05) במספר ההשוואות.

דוגמה: 4 תקופות → 6 זוגות השוואה → α_מתוקן = 0.05/6 = 0.0083

In [ ]:
# נוצר נתונים סינתטיים של 4 תקופות
np.random.seed(42)
n_per = 20
periods = ['EBA', 'MBA', 'LBA', 'Iron']
# כל תקופה נבדלת מעט מהשאר
period_data = [
    np.random.randn(n_per, 10) + i * 0.3
    for i in range(len(periods))
]

# MANOVA פרמוטציה לכל זוג
from itertools import combinations

pairs = list(combinations(range(len(periods)), 2))
n_comparisons = len(pairs)
alpha_bonferroni = 0.05 / n_comparisons

print(f'מספר השוואות: {n_comparisons}')
print(f'α מתוקן (בונפרוני): {alpha_bonferroni:.4f}')
print()

p_matrix = np.ones((len(periods), len(periods)))

for i, j in pairs:
    Xi = period_data[i]
    Xj = period_data[j]
    X_pair = np.vstack([Xi, Xj])
    g_pair = np.array([periods[i]]*n_per + [periods[j]]*n_per)
    _, p, _ = permutation_manova(X_pair, g_pair, n_perm=499)
    p_matrix[i, j] = p
    p_matrix[j, i] = p
    sig = '**' if p < alpha_bonferroni else ('*' if p < 0.05 else 'n.s.')
    print(f'{periods[i]} vs {periods[j]}: p={p:.4f} {sig}')

print()
print(f'(** = מובהק לאחר בונפרוני; * = מובהק ללא תיקון; n.s. = לא מובהק)')

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(p_matrix, cmap='RdYlGn_r', vmin=0, vmax=0.1)
ax.set_xticks(range(len(periods)))
ax.set_yticks(range(len(periods)))
ax.set_xticklabels(periods, fontsize=11)
ax.set_yticklabels(periods, fontsize=11)

for i in range(len(periods)):
    for j in range(len(periods)):
        if i != j:
            p_val_ij = p_matrix[i, j]
            sig = '**' if p_val_ij < alpha_bonferroni else ('*' if p_val_ij < 0.05 else '')
            txt = f'{p_val_ij:.3f}{sig}'
            ax.text(j, i, txt, ha='center', va='center', fontsize=9,
                    color='white' if p_val_ij < 0.03 else 'black')
        else:
            ax.text(j, i, '-', ha='center', va='center', fontsize=12, color='gray')

plt.colorbar(im, ax=ax, label='p-value')
ax.set_title(rtl('מפת חום: p-values בין תקופות (MANOVA הסתברותי)'), fontsize=12)
plt.tight_layout()
plt.show()

print(f'ירוק = לא מובהק, אדום = מובהק מאוד')
print(f'α בונפרוני = {alpha_bonferroni:.4f}')

## סיכום

- **MANOVA הסתברותי**: ויזואליזציה של התפלגות ה-F עוזרת להבין את המשמעות
- **אלומטריה**: `scipy.stats.linregress` עם רווח בטחון 95%
- **בונפרוני**: α_מתוקן = α / מספר_השוואות
- **מפת חום**: דרך נוחה להציג תוצאות השוואות מרובות

**שאלות לחשיבה:**
1. מדוע תיקון בונפרוני הכרחי בהשוואות מרובות?
2. האם יש דרכים אחרות לתקן עבור השוואות מרובות?
3. מה ניתן ללמוד מגרף האלומטריה על תהליך ייצור הקרדומות?